In [ ]:
##CRM export: high-churn-risk segment with SHAP-based retention action codes##

In [2]:
import numpy
import pandas
import shap
import pyarrow

In [3]:
import pandas as pd

In [73]:
data = pd.read_csv(
    r"C:\Users\sabsm\Documents\Amdex DS Group Project Materials\cleaned_online_retail.csv\cleaned_online_retail.csv",
    low_memory=False
)

In [60]:
data.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'CustomerID', 'Country', 'Revenue'],
      dtype='object')

In [61]:
data.isnull().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
CustomerID     228489
Country             0
Revenue             0
dtype: int64

In [43]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007914 entries, 0 to 1007913
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   InvoiceNo    1007914 non-null  object 
 1   StockCode    1007914 non-null  object 
 2   Description  1007914 non-null  object 
 3   Quantity     1007914 non-null  int64  
 4   InvoiceDate  1007914 non-null  object 
 5   Price        1007914 non-null  float64
 6   CustomerID   779425 non-null   float64
 7   Country      1007914 non-null  object 
 8   Revenue      1007914 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 69.2+ MB


In [74]:
def categorize_product(desc):

    desc = str(desc).lower()

    # Storage & Bags
    if (
        "bag" in desc or
        "basket" in desc or
        "box" in desc
    ):
        return "Storage & Bags"

    # Kitchen
    elif (
        "glass" in desc or
        "jar" in desc or
        "plate" in desc or
        "bowl" in desc or
        "kitchen" in desc or
        "mug" in desc or
        "cup" in desc
    ):
        return "Kitchen"

    # Home Decor
    elif (
        "heart" in desc or
        "holder" in desc or
        "candle" in desc or
        "decor" in desc or
        "vase" in desc
    ):
        return "Home Decor"

    # Kids
    elif (
        "toy" in desc or
        "child" in desc or
        "kids" in desc or
        "doll" in desc
    ):
        return "Kids"

    # Seasonal
    elif (
        "christmas" in desc or
        "xmas" in desc or
        "holiday" in desc
    ):
        return "Seasonal"

    # Stationery
    elif (
        "pen" in desc or
        "paper" in desc or
        "card" in desc or
        "notebook" in desc
    ):
        return "Stationery"

    # Lighting
    elif (
        "light" in desc or
        "lamp" in desc
    ):
        return "Lighting"

    # Textile
    elif (
        "fabric" in desc or
        "blanket" in desc or
        "cushion" in desc
    ):
        return "Textile"

    # Photo & Frames
    elif (
        "frame" in desc or
        "photo" in desc
    ):
        return "Photo & Frames"

    # Clocks
    elif "clock" in desc:
        return "Clocks"

    # Garden
    elif (
        "flower" in desc or
        "garden" in desc
    ):
        return "Garden"

    # Furniture & Decor
    elif (
        "wooden" in desc or
        "metal" in desc
    ):
        return "Furniture & Decor"

    # Default
    else:
        return "Other"

In [75]:
df = data.dropna(subset=['CustomerID']).copy()

df['CustomerID'] = df['CustomerID'].astype(int)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Category'] = df['Description'].apply(categorize_product)

In [76]:
snapshot = df['InvoiceDate'].max()

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm = rfm.reset_index()

In [79]:
category_matrix = pd.crosstab(df['CustomerID'], df['Category'])

In [80]:
X = rfm.merge(category_matrix, left_on='CustomerID', right_index=True)

In [81]:
X['Churn'] = (X['Recency'] > 90).astype(int)

In [82]:
from xgboost import XGBClassifier

features = X.drop(columns=['CustomerID','Churn'])

model = XGBClassifier(eval_metric='logloss')

model.fit(features, X['Churn'])

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [83]:
X['churn_risk'] = model.predict_proba(features)[:,1]

In [84]:
X['segment'] = X['churn_risk'].apply(
    lambda x: 'High Risk' if x > 0.7 else
              'Medium Risk' if x > 0.4 else
              'Low Risk'
)

In [85]:
importance_df = pd.DataFrame({
    'feature': features.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

In [86]:
X['top_reason'] = importance_df.iloc[0]['feature']

In [87]:
def crm_action(row):
    if row['churn_risk'] > 0.8:
        return "URGENT CALL + DISCOUNT OFFER"
    elif row['churn_risk'] > 0.5:
        return "EMAIL CAMPAIGN + COUPON"
    else:
        return "LOYALTY ENGAGEMENT"

X['action_code'] = X.apply(crm_action, axis=1)

In [88]:
crm_export = X[[
    'CustomerID',
    'Recency',
    'Frequency',
    'Monetary',
    'churn_risk',
    'segment',
    'top_reason',
    'action_code'
]]

crm_export.to_csv("CRM_OUTPUT.csv", index=False)

In [89]:
import os

os.listdir()

['.anaconda',
 '.bash_history',
 '.cache',
 '.conda',
 '.condarc',
 '.continuum',
 '.gitconfig',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.matplotlib',
 '.ms-ad',
 '.redhat',
 '.spyder-py3',
 '.vscode',
 '3D Objects',
 'Aggregate Functions.ipynb',
 'anaconda3',
 'anki-23.12.1-windows-qt6.exe',
 'AppData',
 'Application Data',
 'Contacts',
 'Cookies',
 'CRM Export.ipynb',
 'CRM_OUTPUT.csv',
 'Customer segmentation & RFM intelligence.ipynb',
 'Data  engineering Notes.docx',
 'Desktop',
 'Documents',
 'Downloads',
 'EDA & Feature Engineering_NeuralRetail.ipynb',
 'Favorites',
 'IntelGraphicsProfiles',
 'Links',
 'Local Settings',
 'MSTeams-x64.msix',
 'Music',
 'My Documents',
 'MyFirstJavaProgram.java',
 'NetHood',
 'NTUSER.DAT',
 'ntuser.dat.LOG1',
 'ntuser.dat.LOG2',
 'NTUSER.DAT{9664df44-acba-11f0-aad4-c206250391fa}.TxR.0.regtrans-ms',
 'NTUSER.DAT{9664df44-acba-11f0-aad4-c206250391fa}.TxR.1.regtrans-ms',
 'NTUSER.DAT{9664df44-acba-11f0-aad4-c206250391fa}.TxR.2.regtrans-ms',

In [90]:
import os
print(os.getcwd())

C:\Users\sabsm


In [91]:
crm_export.head()

,CustomerID,Recency,Frequency,Monetary,churn_risk,segment,top_reason,action_code
0,12346,325,12,77556.46,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
1,12347,1,8,4921.53,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
2,12348,74,5,2019.40,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
3,12349,18,4,4428.69,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
4,12350,309,1,334.40,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER


In [92]:
display(crm_export)

,CustomerID,Recency,Frequency,Monetary,churn_risk,segment,top_reason,action_code
0,12346,325,12,77556.46,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
1,12347,1,8,4921.53,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
2,12348,74,5,2019.40,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
3,12349,18,4,4428.69,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
4,12350,309,1,334.40,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
...,...,...,...,...,...,...,...,...
5873,18283,3,22,2664.90,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
5874,18284,431,1,461.68,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
5875,18285,660,1,427.00,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
5876,18286,476,2,1296.43,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER


In [93]:
crm_export.shape

(5878, 8)

In [95]:
crm_export.to_csv("CRM_OUTPUT.csv", index=False)
print("CRM file created successfully!")
crm_export.head()

CRM file created successfully!


,CustomerID,Recency,Frequency,Monetary,churn_risk,segment,top_reason,action_code
0,12346,325,12,77556.46,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
1,12347,1,8,4921.53,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
2,12348,74,5,2019.40,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
3,12349,18,4,4428.69,0.000341,Low Risk,Recency,LOYALTY ENGAGEMENT
4,12350,309,1,334.40,0.999669,High Risk,Recency,URGENT CALL + DISCOUNT OFFER
